<a href="https://colab.research.google.com/github/lohaniSatwik/steam-games-data-mining/blob/master/Code/section6_threshold_experiment_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Section 6 — Threshold Experiment (Steam Natural Boundaries)
**IE500 Data Mining | Team 9 – Brewed Clusters**

> **Google Colab notebook.** Run all cells top to bottom.

## Research Question
Does changing the class labeling threshold change model performance?  
The original project used arbitrary thresholds. This experiment uses **Steam's own natural category boundaries**.

## Threshold Comparison

| Class | Original (0.5) | This Experiment (0.4) |
|-------|---------------|----------------------|
| **Good**  | positive_ratio ≥ 75%  | positive_ratio ≥ 70% |
| **Mixed** | 50% – 74%             | 40% – 69%            |
| **Bad**   | < 50%                 | < 40%                |

The 0.4-threshold aligns with **Steam's own review display**:  
- Steam shows "Mostly Positive" at ≥ 70% — so we use 70% as the Good cutoff  
- Steam shows "Mixed" from 40–69% — so Mixed range starts at 40%  
- Below 40% is "Mostly Negative" in Steam's own language

### Impact on class distribution

| Class | Original | Experiment | Change |
|-------|----------|------------|--------|
| Good  | 63.3%    | 71.4%      | +8.1%  |
| Mixed | 28.3%    | 24.3%      | −4.0%  |
| Bad   | 8.4%     | 4.3%       | −4.1%  |

More games now fall into **Good** (the lower 70% bar is easier to clear).  
**Bad** class shrinks almost in half — making it even harder to classify correctly.

## Model
- **Logistic Regression** — same as the original baseline for a direct apples-to-apples comparison
- `class_weight='balanced'`, L2 regularisation, lbfgs solver
- **Outer CV:** 5-fold stratified | **Inner CV:** 3-fold `GridSearchCV` over `C`
- **Metric:** Macro F1 (treats all 3 classes equally regardless of size)
- **Reference baseline (original LR):** Macro F1 = **0.4355**

In [ ]:
import os
if not os.path.exists('steam-games-data-mining'):
    !git clone https://github.com/lohaniSatwik/steam-games-data-mining.git
else:
    !git -C steam-games-data-mining pull
DATA_DIR = 'steam-games-data-mining/Datasets'

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from collections import Counter
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
CLASS_ORDER  = ['Good', 'Mixed', 'Bad']
CLASS_COLORS = {'Good': 'steelblue', 'Mixed': 'sandybrown', 'Bad': 'salmon'}

# Original LR baseline (Good>=75%, Mixed 50-74%, Bad<50%)
BASELINE_ORIGINAL_LR = 0.4355

print('Libraries loaded.')

In [ ]:
# Load the 0.4-threshold datasets (Steam natural category boundaries)
train = pd.read_csv(f'{DATA_DIR}/train_multiclass_0.4.csv')
test  = pd.read_csv(f'{DATA_DIR}/test_multiclass_0.4.csv')

X_train = train.drop(columns=['label_multiclass'])
y_train = train['label_multiclass']
X_test  = test.drop(columns=['label_multiclass'])
y_test  = test['label_multiclass']

print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'Features: {X_train.shape[1]}')
print()
print('Class distribution (train) — 0.4-threshold:')
vc = y_train.value_counts()
for cls in CLASS_ORDER:
    print(f'  {cls:6s}: {vc[cls]:6,d}  ({vc[cls]/len(y_train)*100:.1f}%)')

print()
print('Class distribution (train) — original (0.5-threshold) for comparison:')
print('  Good  : 28,671  (63.3%)')
print('  Mixed : 12,834  (28.3%)')
print('  Bad   :  3,819  ( 8.4%)')

## Nested Cross-Validation

- **Outer loop** (5 folds) — gives an unbiased estimate of generalisation performance
- **Inner loop** (3-fold `GridSearchCV`) — selects the best `C` without touching the outer validation fold
- `class_weight='balanced'` re-weights the loss function to correct for class imbalance
- Same setup as original LR baseline — only the data labels change

Expected runtime on Colab: **~20–30 minutes**

In [ ]:
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

param_grid = {'C': [0.01, 0.1, 1, 10], 'penalty': ['l2'], 'solver': ['lbfgs']}

outer_scores    = []
best_params_log = []

print('Running 5-fold nested CV (inner 3-fold GridSearchCV)...\n')

for fold, (tr_idx, val_idx) in tqdm(
        enumerate(outer_cv.split(X_train, y_train), 1),
        total=5, desc='Outer folds'):

    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    base_clf = LogisticRegression(
        class_weight='balanced', random_state=RANDOM_STATE, max_iter=500
    )
    gs = GridSearchCV(
        base_clf, param_grid,
        cv=inner_cv, scoring='f1_macro',
        n_jobs=-1, refit=True
    )
    gs.fit(X_tr, y_tr)

    y_pred = gs.predict(X_val)
    f1 = f1_score(y_val, y_pred, average='macro')
    outer_scores.append(f1)
    best_params_log.append(gs.best_params_)

    print(f'  Fold {fold} | Macro F1: {f1:.4f} | {gs.best_params_}')

print(f'\nNested CV  (0.4-threshold) →  Macro F1: {np.mean(outer_scores):.4f} ± {np.std(outer_scores):.4f}')
print(f'Original LR (0.5-threshold) →  Macro F1: {BASELINE_ORIGINAL_LR:.4f}')
print(f'Difference                  →  {np.mean(outer_scores) - BASELINE_ORIGINAL_LR:+.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
folds = list(range(1, 6))
ax.bar(folds, outer_scores, color='teal', edgecolor='white', alpha=0.85)
ax.axhline(np.mean(outer_scores), color='crimson', linestyle='--', linewidth=1.5,
           label=f'LR 0.4 Mean = {np.mean(outer_scores):.4f}')
ax.axhline(BASELINE_ORIGINAL_LR, color='steelblue', linestyle='--', linewidth=1.5,
           label=f'LR 0.5 Baseline = {BASELINE_ORIGINAL_LR:.4f}')
ax.axhline(np.mean(outer_scores) + np.std(outer_scores), color='crimson',
           linestyle=':', linewidth=1, alpha=0.6)
ax.axhline(np.mean(outer_scores) - np.std(outer_scores), color='crimson',
           linestyle=':', linewidth=1, alpha=0.6,
           label=f'±1 std = {np.std(outer_scores):.4f}')
ax.set_xlabel('Outer Fold')
ax.set_ylabel('Macro F1')
ax.set_title('LR Threshold Experiment — Nested CV Macro F1 per Fold\n(0.4 vs 0.5 boundary)')
ax.set_xticks(folds)
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
print('Best hyperparameters across outer folds:')
param_counts = Counter([str(p) for p in best_params_log])
for params, count in param_counts.most_common():
    print(f'  {count:2d} fold(s): {params}')

## Final Model

Re-run `GridSearchCV` on the **full training set** to select the best hyperparameters, then evaluate on the held-out test set **once**.

In [ ]:
print('Fitting final model on full training set...\n')

final_gs = GridSearchCV(
    LogisticRegression(
        class_weight='balanced', random_state=RANDOM_STATE, max_iter=2000
    ),
    param_grid,
    cv=inner_cv, scoring='f1_macro',
    n_jobs=-1, refit=True
)
final_gs.fit(X_train, y_train)

print(f'Best params      : {final_gs.best_params_}')
print(f'Best inner CV F1 : {final_gs.best_score_:.4f}')

## Test Set Evaluation

> Evaluate on `test_multiclass_0.4.csv` **once only** — this is the final performance number.

In [ ]:
final_model = final_gs.best_estimator_
y_pred_test = final_model.predict(X_test)

test_macro_f1 = f1_score(y_test, y_pred_test, average='macro')
print(f'Test set Macro F1 (0.4-threshold) : {test_macro_f1:.4f}')
print(f'Original LR Macro F1 (0.5-thresh) : {BASELINE_ORIGINAL_LR:.4f}')
print(f'Difference                         : {test_macro_f1 - BASELINE_ORIGINAL_LR:+.4f}\n')
print('Classification Report (Test Set):')
print(classification_report(y_test, y_pred_test, labels=CLASS_ORDER, target_names=CLASS_ORDER))

In [ ]:
cm = confusion_matrix(y_test, y_pred_test, labels=CLASS_ORDER)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_ORDER)
disp.plot(ax=ax, colorbar=False, cmap='GnBu')
ax.set_title('LR Threshold Experiment — Confusion Matrix (Test Set)\n(0.4 boundary: Good≥70%, Mixed 40–69%, Bad<40%)', pad=12)
plt.tight_layout()
plt.show()

## Class Distribution Comparison Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Original (0.5 threshold)
orig_counts = {'Good': 28671, 'Mixed': 12834, 'Bad': 3819}
orig_total  = sum(orig_counts.values())
axes[0].bar(CLASS_ORDER,
            [orig_counts[c] for c in CLASS_ORDER],
            color=[CLASS_COLORS[c] for c in CLASS_ORDER], alpha=0.85)
axes[0].set_title('Original (0.5-threshold)\nGood≥75%, Mixed 50–74%, Bad<50%')
axes[0].set_ylabel('Count')
for i, cls in enumerate(CLASS_ORDER):
    axes[0].text(i, orig_counts[cls] + 200,
                 f'{orig_counts[cls]/orig_total*100:.1f}%', ha='center', fontsize=11)

# Experiment (0.4 threshold)
vc = y_train.value_counts()
exp_total = len(y_train)
axes[1].bar(CLASS_ORDER,
            [vc[c] for c in CLASS_ORDER],
            color=[CLASS_COLORS[c] for c in CLASS_ORDER], alpha=0.85)
axes[1].set_title('Experiment (0.4-threshold)\nGood≥70%, Mixed 40–69%, Bad<40%')
axes[1].set_ylabel('Count')
for i, cls in enumerate(CLASS_ORDER):
    axes[1].text(i, vc[cls] + 200,
                 f'{vc[cls]/exp_total*100:.1f}%', ha='center', fontsize=11)

plt.suptitle('Class Distribution: Original vs 0.4-Threshold Experiment', fontsize=13)
plt.tight_layout()
plt.show()

## Feature Importance

Logistic Regression (One-vs-Rest) produces one coefficient per feature per class.  
Comparing these to the original LR coefficients shows which features are sensitive to the threshold change.

In [ ]:
feature_names = X_train.columns.tolist()
coef = final_model.coef_   # shape: (n_classes, n_features)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, cls in enumerate(final_model.classes_):
    color = CLASS_COLORS[cls]
    coef_series = pd.Series(coef[i], index=feature_names)
    top_idx = coef_series.abs().nlargest(15).index
    coef_series[top_idx].sort_values().plot(
        kind='barh', ax=axes[i], color=color, alpha=0.85
    )
    axes[i].axvline(0, color='black', linewidth=0.8)
    axes[i].set_title(f'Top 15 Coefficients — {cls}')
    axes[i].set_xlabel('Coefficient')

plt.suptitle('LR Threshold Experiment — Feature Importance (OvR Coefficients, 0.4-threshold)', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

## Results Summary

### Performance

| Metric | 0.4-Threshold (this experiment) | Original 0.5-Threshold (LR baseline) |
|--------|----------------------------------|--------------------------------------|
| Nested CV Macro F1 (mean ± std) | — | **0.4385 ± 0.0034** |
| Test set Macro F1 | — | **0.4355** |
| Best params | — | C=0.1, L2, lbfgs |

**Per-class F1 (test set):**

| Class | 0.4-Threshold |  | Original 0.5-Threshold |
|-------|:---:|---|:---:|
| Good  | — | vs | 0.70 |
| Mixed | — | vs | 0.33 |
| Bad   | — | vs | 0.28 |

> Fill in the left column after running.

### Interpretation

**What to look for:**
- If Macro F1 **increases**: the 0.4-threshold creates cleaner decision boundaries — the "Mixed" zone at 40–69% is more internally coherent than the 50–74% zone, making it easier to separate from Good and Bad.
- If Macro F1 **decreases**: the much smaller Bad class (4.3% vs 8.4%) is even harder to detect with only ~1,963 training examples. The majority Good class (71.4%) dominates even more, hurting Macro F1.
- **Bad class F1** is the most informative metric: does lowering the Bad threshold (to <40%) produce a more internally homogeneous class that's easier to identify?
- **Mixed class F1**: the 40–69% range is wider (30 pp) vs the original 50–74% (25 pp), but it also excludes the ambiguous 40–49% zone from "Bad". May improve or hurt depending on data distribution.

**Academic relevance:**  
This experiment tests whether the labeling convention (an analytical choice) materially affects predictive performance.  
If results are similar across both thresholds, the finding supports robustness of the approach.  
If results differ significantly, it highlights how label definition choices affect ML outcomes — a methodological insight worth reporting.